# Auto-MQT Colab Pipeline

Run heavy Auto-MQT stages here: oracle labeling, frozen feature extraction, router training, and evaluation.

## 1) Runtime Check

Set runtime to GPU before continuing.

In [1]:
import torch
print('cuda_available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'Please switch Colab runtime to GPU.'

cuda_available: False
device: cpu


AssertionError: Please switch Colab runtime to GPU.

## 2) Repo Setup

This cell mounts Drive and auto-resolves your repo path.

Important: Colab mount exposes `MyDrive` (and shared drives), not the Drive web `Computers > My Laptop` tree directly. If your project lives under `Computers`, add a shortcut or copy of `Auto-MQT-` into `MyDrive` once, then this notebook can find it automatically.

In [2]:
import os
import sys
from pathlib import Path
from google.colab import drive

MOUNT_DIR = Path('/content/drive')
drive.mount(str(MOUNT_DIR), force_remount=False)

def resolve_auto_mqt_repo() -> Path:
    explicit = os.environ.get('AUTO_MQT_REPO')
    candidates = []
    if explicit:
        candidates.append(Path(explicit))

    candidates.extend([
        MOUNT_DIR / 'MyDrive' / 'Github Repos' / 'Auto-MQT-',
        MOUNT_DIR / 'MyDrive' / 'Colab Notebooks' / 'Auto-MQT-',
        Path('/content/Auto-MQT-'),
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        'Auto-MQT repo not found. Set AUTO_MQT_REPO, or place/shortcut Auto-MQT- under /content/drive/MyDrive.'
    )

AUTO_MQT_REPO = resolve_auto_mqt_repo()
MQT_REPO = Path('/content/MQT-LLaVA')
print('auto_mqt_repo:', AUTO_MQT_REPO)
print('drive_root:', MOUNT_DIR / 'MyDrive')

%cd /content
!test -d MQT-LLaVA || git clone https://github.com/gordonhu608/MQT-LLaVA.git

%cd {AUTO_MQT_REPO}
!python -m pip install --upgrade pip
# Remove preinstalled package that requires newer transformers than MQT-LLaVA pins.
!pip uninstall -y sentence-transformers

# Colab-specific requirements exclude Jupyter stack managed by google-colab.
!pip install -r requirements_colab.txt
!pip install -e /content/MQT-LLaVA --no-deps

sys.path.insert(0, str(AUTO_MQT_REPO / 'src'))
print('mqt_repo:', MQT_REPO)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
auto_mqt_repo: /content/drive/Othercomputers/My Laptop/Auto-MQT-
drive_root: /content/drive/MyDrive
/content
/content/drive/Othercomputers/My Laptop/Auto-MQT-
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Obtaining file:///content/MQT-LLaVA
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llava (pyproject.toml) ... done
  Created wheel for llava: filename=llava-1.0.1-0.editable-py3-none-any.whl size=10552 sha256=12a5141947bf10d1841d99af4657c9479df10d59037fd6721902ff00a12464f7
  Stored in directory: /tmp/pip-ephem-wheel-cache-7t0h6z2u/wheels/fe/b9/de/b105e1f4dbec25ca50a14ba951b135dc129eabccb934bd828b
Successfully built llava
  Attempting unins

## 3) Environment Variables

In [ ]:
import os
from pathlib import Path
import yaml


def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        print(f'.env not found at {env_path}')
        return
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_env_file(Path(AUTO_MQT_REPO) / '.env')

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    os.environ['HF_HUB_TOKEN'] = hf_token
    print('HF token loaded from environment/.env')
else:
    print('HF token not found in environment/.env (public models may still work)')

os.environ['MQT_LLAVA_REPO'] = '/content/MQT-LLaVA'
os.environ['MQT_LLAVA_MODEL_PATH'] = 'gordonhu/MQT-LLaVA-7b'
os.environ['MQT_LLAVA_BACKEND'] = 'persistent'
os.environ['MQT_LLAVA_DEVICE_MAP'] = 'auto'
os.environ['MQT_LLAVA_OFFLOAD_FOLDER'] = '/content/offload'

# Memory controls. Keep 8-bit off by default; set to 1 if you still hit OOM.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('MQT_LLAVA_LOAD_8BIT', '0')
os.environ.setdefault('MQT_LLAVA_LOAD_4BIT', '0')
os.environ.setdefault('AUTO_MQT_SUPPRESS_LOAD_WARNINGS', '1')

# Dataset toggle: 1 = include TextVQA (default/current behavior), 0 = exclude TextVQA end-to-end.
os.environ.setdefault('INCLUDE_TEXTVQA', '0')

INCLUDE_TEXTVQA = int(os.environ.get('INCLUDE_TEXTVQA', '1'))
if INCLUDE_TEXTVQA not in (0, 1):
    raise ValueError(f'INCLUDE_TEXTVQA must be 0 or 1, got {INCLUDE_TEXTVQA}')

EXCLUDE_DATASETS_ARGS = '' if INCLUDE_TEXTVQA == 1 else '--exclude-datasets textvqa'
PREPARE_DATASETS_ARGS = '' if INCLUDE_TEXTVQA == 1 else '--datasets vqav2 gqa scienceqa_img'

RUN_TAG = 'proposal_4ds' if INCLUDE_TEXTVQA == 1 else 'proposal_no_textvqa'
RESULTS_DIR = 'results' if INCLUDE_TEXTVQA == 1 else f'results/{RUN_TAG}'
CHECKPOINT_DIR = 'checkpoints' if INCLUDE_TEXTVQA == 1 else f'checkpoints/{RUN_TAG}'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

if INCLUDE_TEXTVQA == 1:
    TRAIN_MANIFEST = 'data/manifests/train_proposal_balanced.jsonl'
    EVAL_MANIFEST = 'data/manifests/eval_proposal_balanced.jsonl'
    ORACLE_TRAIN_MANIFEST = 'data/manifests/oracle_train_proposal.jsonl'
    ORACLE_EVAL_MANIFEST = 'data/manifests/oracle_eval_proposal.jsonl'
    TRAIN_FEATURES_MANIFEST = 'data/manifests/oracle_train_proposal_with_features.jsonl'
    EVAL_FEATURES_MANIFEST = 'data/manifests/eval_proposal_with_features.jsonl'
    BREAKDOWN_PREFIX = 'proposal_4ds'
else:
    TRAIN_MANIFEST = 'data/manifests/train_proposal_balanced_no_textvqa.jsonl'
    EVAL_MANIFEST = 'data/manifests/eval_proposal_balanced_no_textvqa.jsonl'
    ORACLE_TRAIN_MANIFEST = 'data/manifests/oracle_train_proposal_no_textvqa.jsonl'
    ORACLE_EVAL_MANIFEST = 'data/manifests/oracle_eval_proposal_no_textvqa.jsonl'
    TRAIN_FEATURES_MANIFEST = 'data/manifests/oracle_train_proposal_with_features_no_textvqa.jsonl'
    EVAL_FEATURES_MANIFEST = 'data/manifests/eval_proposal_with_features_no_textvqa.jsonl'
    BREAKDOWN_PREFIX = 'proposal_3ds_no_textvqa'

RESULT_FIXED_36 = f'{RESULTS_DIR}/fixed_36.jsonl'
RESULT_FIXED_64 = f'{RESULTS_DIR}/fixed_64.jsonl'
RESULT_FIXED_144 = f'{RESULTS_DIR}/fixed_144.jsonl'
RESULT_FIXED_256 = f'{RESULTS_DIR}/fixed_256.jsonl'
RESULT_ROUTER_PROMPT = f'{RESULTS_DIR}/router_prompt.jsonl'
RESULT_ROUTER_IMAGE = f'{RESULTS_DIR}/router_image.jsonl'
RESULT_ROUTER_MULTIMODAL = f'{RESULTS_DIR}/router_multimodal.jsonl'
RESULT_ROUTER_CROSS_ATTN = f'{RESULTS_DIR}/router_cross_attention.jsonl'
SUMMARY_CSV = f'{RESULTS_DIR}/summary.csv'
FINAL_SUMMARY_MD = f'{RESULTS_DIR}/final_summary.md'

CKPT_ROUTER_PROMPT = f'{CHECKPOINT_DIR}/router_prompt.pt'
CKPT_ROUTER_IMAGE = f'{CHECKPOINT_DIR}/router_image.pt'
CKPT_ROUTER_MULTIMODAL = f'{CHECKPOINT_DIR}/router_multimodal.pt'
CKPT_ROUTER_CROSS_ATTN = f'{CHECKPOINT_DIR}/router_cross_attention.pt'

BASE_DATASET_CONFIG = Path(AUTO_MQT_REPO) / 'configs/datasets_proposal_balanced.yaml'
PREPARE_CONFIG = str(BASE_DATASET_CONFIG)
if INCLUDE_TEXTVQA == 0:
    tmp_config = Path(AUTO_MQT_REPO) / 'configs/datasets_proposal_balanced_no_textvqa_tmp.yaml'
    cfg = yaml.safe_load(BASE_DATASET_CONFIG.read_text(encoding='utf-8'))
    cfg['output']['train_manifest'] = Path(TRAIN_MANIFEST).name
    cfg['output']['eval_manifest'] = Path(EVAL_MANIFEST).name
    tmp_config.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    PREPARE_CONFIG = str(tmp_config)

RUN_CLEANUP_PATHS = [
    TRAIN_MANIFEST,
    EVAL_MANIFEST,
    ORACLE_TRAIN_MANIFEST,
    ORACLE_EVAL_MANIFEST,
    TRAIN_FEATURES_MANIFEST,
    EVAL_FEATURES_MANIFEST,
    RESULT_FIXED_36,
    RESULT_FIXED_64,
    RESULT_FIXED_144,
    RESULT_FIXED_256,
    RESULT_ROUTER_PROMPT,
    RESULT_ROUTER_IMAGE,
    RESULT_ROUTER_MULTIMODAL,
    RESULT_ROUTER_CROSS_ATTN,
    SUMMARY_CSV,
    FINAL_SUMMARY_MD,
    CKPT_ROUTER_PROMPT,
    CKPT_ROUTER_IMAGE,
    CKPT_ROUTER_MULTIMODAL,
    CKPT_ROUTER_CROSS_ATTN,
]

for key in [
    'MQT_LLAVA_REPO',
    'MQT_LLAVA_MODEL_PATH',
    'MQT_LLAVA_BACKEND',
    'MQT_LLAVA_DEVICE_MAP',
    'MQT_LLAVA_OFFLOAD_FOLDER',
    'MQT_LLAVA_LOAD_8BIT',
    'MQT_LLAVA_LOAD_4BIT',
    'AUTO_MQT_SUPPRESS_LOAD_WARNINGS',
    'PYTORCH_CUDA_ALLOC_CONF',
    'INCLUDE_TEXTVQA',
]:
    print(key, '=', os.environ[key])

print('RUN_TAG =', RUN_TAG)
print('PREPARE_CONFIG =', PREPARE_CONFIG)
print('EXCLUDE_DATASETS_ARGS =', EXCLUDE_DATASETS_ARGS or '(none)')
print('PREPARE_DATASETS_ARGS =', PREPARE_DATASETS_ARGS or '(none)')
print('RESULTS_DIR =', RESULTS_DIR)
print('CHECKPOINT_DIR =', CHECKPOINT_DIR)
print('TRAIN_MANIFEST =', TRAIN_MANIFEST)
print('EVAL_MANIFEST =', EVAL_MANIFEST)


HF token loaded from environment/.env
MQT_LLAVA_REPO = /content/MQT-LLaVA
MQT_LLAVA_MODEL_PATH = gordonhu/MQT-LLaVA-7b
MQT_LLAVA_BACKEND = persistent
MQT_LLAVA_DEVICE_MAP = auto
MQT_LLAVA_OFFLOAD_FOLDER = /content/offload
MQT_LLAVA_LOAD_8BIT = 0
MQT_LLAVA_LOAD_4BIT = 0
AUTO_MQT_SUPPRESS_LOAD_WARNINGS = 1
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True


## 3.1 Backend Sanity Check

Make sure we are running MQT backend (not the local smoke backend).

In [4]:
%cd {AUTO_MQT_REPO}
import os
from pathlib import Path

required = [
    'MQT_LLAVA_REPO',
    'MQT_LLAVA_MODEL_PATH',
    'MQT_LLAVA_BACKEND',
]
for key in required:
    assert os.environ.get(key), f'missing required env: {key}'

assert Path(os.environ['MQT_LLAVA_REPO']).exists(), 'MQT_LLAVA_REPO path does not exist'
assert os.environ['MQT_LLAVA_BACKEND'] in {'persistent', 'eval'}, 'unexpected backend mode'

print('backend_mode:', os.environ['MQT_LLAVA_BACKEND'])
print('model_path:', os.environ['MQT_LLAVA_MODEL_PATH'])
print('repo_path_exists:', Path(os.environ['MQT_LLAVA_REPO']).exists())
print('sanity_check: config-only check passed (model not loaded)')



/content/drive/Othercomputers/My Laptop/Auto-MQT-
backend_mode: persistent
model_path: gordonhu/MQT-LLaVA-7b
repo_path_exists: True
sanity_check: config-only check passed (model not loaded)


## 4) Build / Verify Manifests

Set `INCLUDE_TEXTVQA=1` (default) for 4-dataset run, or `INCLUDE_TEXTVQA=0` for no-TextVQA run.
No-TextVQA mode writes separate manifests/results/checkpoints so existing 4-dataset outputs are preserved.


In [5]:
%cd {AUTO_MQT_REPO}
from pathlib import Path

for rel in RUN_CLEANUP_PATHS:
    path = Path(rel)
    if path.exists():
        path.unlink()
        print('removed', path)

!python src/prepare_datasets.py --config {PREPARE_CONFIG} --prompt-style none --strict-datasets {PREPARE_DATASETS_ARGS}
!python src/verify_manifest.py --manifest {TRAIN_MANIFEST}
!python src/verify_manifest.py --manifest {EVAL_MANIFEST}


/content/drive/Othercomputers/My Laptop/Auto-MQT-
README.md: 100% 509/509 [00:00<00:00, 4.00MB/s]
data/train-00000-of-00010-5a836b85e3ef0e(…): 100% 480M/480M [00:03<00:00, 158MB/s] 
data/train-00001-of-00010-f9a613ba4b6c3c(…): 100% 485M/485M [00:02<00:00, 186MB/s] 
data/train-00002-of-00010-aae550539cfec7(…): 100% 477M/477M [00:02<00:00, 216MB/s]
data/train-00003-of-00010-c21954183685ea(…): 100% 485M/485M [00:08<00:00, 57.6MB/s]
data/train-00004-of-00010-260888b7a6817c(…): 100% 488M/488M [00:09<00:00, 53.0MB/s]
data/train-00005-of-00010-e44162ecab38be(…): 100% 488M/488M [00:02<00:00, 187MB/s] 
data/train-00006-of-00010-611f74b1b98566(…): 100% 490M/490M [00:04<00:00, 102MB/s] 
data/train-00007-of-00010-1fb96beac51a30(…): 100% 483M/483M [00:02<00:00, 172MB/s] 
data/train-00008-of-00010-ea49c35e2ed677(…): 100% 486M/486M [00:09<00:00, 49.5MB/s]
data/train-00009-of-00010-b4a16fbbbbec39(…): 100% 492M/492M [00:02<00:00, 204MB/s] 
Generating train split: 100% 10000/10000 [00:19<00:00, 518.60 e

## 5) Oracle Labels (Heavy, Resumable)

Rerun the same command after disconnects; it skips completed `example_id`s.

In [6]:
%cd {AUTO_MQT_REPO}
!python src/oracle_labeling.py --data {TRAIN_MANIFEST} --out {ORACLE_TRAIN_MANIFEST} --budgets 36 64 144 256 --score-key dataset_score --zero-score-budget 64 --tolerance 0.02 --prompt-style short {EXCLUDE_DATASETS_ARGS}
!python src/oracle_diagnostics.py --data {ORACLE_TRAIN_MANIFEST}


/content/drive/Othercomputers/My Laptop/Auto-MQT-
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_

## 6) Frozen Feature Extraction (Heavy)

Writes `prompt_embedding` and `image_embedding` into JSONL.

In [7]:
%cd {AUTO_MQT_REPO}
!python src/extract_router_features.py --data {ORACLE_TRAIN_MANIFEST} --out {TRAIN_FEATURES_MANIFEST} --clip-model openai/clip-vit-large-patch14-336 --batch-size 16 --normalize {EXCLUDE_DATASETS_ARGS}
!python src/extract_router_features.py --data {EVAL_MANIFEST} --out {EVAL_FEATURES_MANIFEST} --clip-model openai/clip-vit-large-patch14-336 --batch-size 16 --normalize {EXCLUDE_DATASETS_ARGS}


/content/drive/Othercomputers/My Laptop/Auto-MQT-
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
rows_total: 2000
rows_pending: 2000
device: cuda
dtype: torch.float16
clip_model: openai/clip-vit-large-patch14-336
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: Fut

## 7) Train Routers

In [8]:
%cd {AUTO_MQT_REPO}
!python src/train_router.py --labels {TRAIN_FEATURES_MANIFEST} --config configs/router_proposal_4budgets.yaml --mode prompt --out {CKPT_ROUTER_PROMPT} {EXCLUDE_DATASETS_ARGS}
!python src/train_router.py --labels {TRAIN_FEATURES_MANIFEST} --config configs/router_proposal_4budgets.yaml --mode image --out {CKPT_ROUTER_IMAGE} {EXCLUDE_DATASETS_ARGS}
!python src/train_router.py --labels {TRAIN_FEATURES_MANIFEST} --config configs/router_proposal_4budgets.yaml --mode multimodal --out {CKPT_ROUTER_MULTIMODAL} {EXCLUDE_DATASETS_ARGS}
!python src/train_router.py --labels {TRAIN_FEATURES_MANIFEST} --config configs/router_proposal_4budgets.yaml --mode cross_attention --out {CKPT_ROUTER_CROSS_ATTN} {EXCLUDE_DATASETS_ARGS}


/content/drive/Othercomputers/My Laptop/Auto-MQT-
override dims from labels: prompt_dim 512 -> 768, image_dim 512 -> 768
class_counts_train: [843, 708, 32, 17]
class_counts_val: [211, 177, 8, 4]
class_weights: [0.0741, 0.0832, 1.3397, 2.503]
epoch=01 objective_loss=-0.3629 val_acc=0.0100 val_score=0.5330 avg_budget=256.00 regret=203.25 under=0.0000 objective=0.4533
epoch=02 objective_loss=-0.4616 val_acc=0.0200 val_score=0.5348 avg_budget=190.56 regret=137.81 under=0.0000 objective=0.4781
epoch=03 objective_loss=-0.5871 val_acc=0.0775 val_score=0.5290 avg_budget=163.80 regret=111.89 under=0.0075 objective=0.4808
epoch=04 objective_loss=-0.6339 val_acc=0.1450 val_score=0.5340 avg_budget=148.00 regret=96.09 under=0.0075 objective=0.4914
epoch=05 objective_loss=-0.6563 val_acc=0.2725 val_score=0.5365 avg_budget=121.11 regret=74.72 under=0.1875 objective=0.4842
epoch=06 objective_loss=-0.6432 val_acc=0.3025 val_score=0.5267 avg_budget=108.34 regret=63.94 under=0.2275 objective=0.4746
epoch

## 8) Evaluate Fixed Budgets + Routers

In [9]:
%cd {AUTO_MQT_REPO}
!python src/evaluate_token_policy.py --data {EVAL_FEATURES_MANIFEST} --fixed-budget 36 --prompt-style short --out {RESULT_FIXED_36} {EXCLUDE_DATASETS_ARGS}
!python src/evaluate_token_policy.py --data {EVAL_FEATURES_MANIFEST} --fixed-budget 64 --prompt-style short --out {RESULT_FIXED_64} {EXCLUDE_DATASETS_ARGS}
!python src/evaluate_token_policy.py --data {EVAL_FEATURES_MANIFEST} --fixed-budget 144 --prompt-style short --out {RESULT_FIXED_144} {EXCLUDE_DATASETS_ARGS}
!python src/evaluate_token_policy.py --data {EVAL_FEATURES_MANIFEST} --fixed-budget 256 --prompt-style short --out {RESULT_FIXED_256} {EXCLUDE_DATASETS_ARGS}

!python src/evaluate_router.py --data {EVAL_FEATURES_MANIFEST} --checkpoint {CKPT_ROUTER_PROMPT} --out {RESULT_ROUTER_PROMPT} {EXCLUDE_DATASETS_ARGS}
!python src/evaluate_router.py --data {EVAL_FEATURES_MANIFEST} --checkpoint {CKPT_ROUTER_IMAGE} --out {RESULT_ROUTER_IMAGE} {EXCLUDE_DATASETS_ARGS}
!python src/evaluate_router.py --data {EVAL_FEATURES_MANIFEST} --checkpoint {CKPT_ROUTER_MULTIMODAL} --out {RESULT_ROUTER_MULTIMODAL} {EXCLUDE_DATASETS_ARGS}
!python src/evaluate_router.py --data {EVAL_FEATURES_MANIFEST} --checkpoint {CKPT_ROUTER_CROSS_ATTN} --out {RESULT_ROUTER_CROSS_ATTN} {EXCLUDE_DATASETS_ARGS}


/content/drive/Othercomputers/My Laptop/Auto-MQT-
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possi

## 9) Optional Oracle on Eval (for regret diagnostics)

Skip if you only need accuracy/tokens/latency.

In [10]:
%cd {AUTO_MQT_REPO}
!python src/oracle_labeling.py --data {EVAL_MANIFEST} --out {ORACLE_EVAL_MANIFEST} --budgets 36 64 144 256 --score-key dataset_score --zero-score-budget 64 --tolerance 0.02 --prompt-style short {EXCLUDE_DATASETS_ARGS}
!python src/oracle_diagnostics.py --data {ORACLE_EVAL_MANIFEST}


/content/drive/Othercomputers/My Laptop/Auto-MQT-
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possi

## 10) Summary Table

Outputs are written to the run-specific folder:
- `INCLUDE_TEXTVQA=1`: `results/` and `checkpoints/`
- `INCLUDE_TEXTVQA=0`: `results/proposal_no_textvqa/` and `checkpoints/proposal_no_textvqa/`


In [11]:
%cd {AUTO_MQT_REPO}
!python src/analyze_results.py   --inputs     fixed_36={RESULT_FIXED_36}     fixed_64={RESULT_FIXED_64}     fixed_144={RESULT_FIXED_144}     fixed_256={RESULT_FIXED_256}     router_prompt={RESULT_ROUTER_PROMPT}     router_image={RESULT_ROUTER_IMAGE}     router_multimodal={RESULT_ROUTER_MULTIMODAL}     router_cross_attention={RESULT_ROUTER_CROSS_ATTN}   --oracle {ORACLE_EVAL_MANIFEST}   --train-manifest {TRAIN_MANIFEST}   --eval-manifest {EVAL_FEATURES_MANIFEST}   --checkpoints     router_prompt={CKPT_ROUTER_PROMPT}     router_image={CKPT_ROUTER_IMAGE}     router_multimodal={CKPT_ROUTER_MULTIMODAL}     router_cross_attention={CKPT_ROUTER_CROSS_ATTN}   --backend-label mqt-persistent   --out-csv {SUMMARY_CSV}   --out-markdown {FINAL_SUMMARY_MD}   {EXCLUDE_DATASETS_ARGS}


/content/drive/Othercomputers/My Laptop/Auto-MQT-
run                    | examples | exact  | relaxed | dataset_score | avg_tokens | avg_latency_s | regret | under_rate
-----------------------+----------+--------+---------+---------------+------------+---------------+--------+-----------
fixed_36               | 600      | 0.5467 | 0.5567  | 0.5387        | 36.00      | 0.271         | 0.00   | 0.4533    
fixed_64               | 600      | 0.5567 | 0.5700  | 0.5475        | 64.00      | 0.285         | 15.31  | 0.0283    
fixed_144              | 600      | 0.5633 | 0.5833  | 0.5532        | 144.00     | 0.305         | 93.04  | 0.0133    
fixed_256              | 600      | 0.5533 | 0.5717  | 0.5428        | 256.00     | 0.354         | 203.55 | 0.0000    
router_prompt          | 600      | 0.5583 | 0.5717  | 0.5485        | 87.21      | 0.293         | 38.95  | 0.0633    
router_image           | 600      | 0.5717 | 0.5867  | 0.5607        | 104.89     | 0.298         | 54.67  | 0

In [5]:
%cd {AUTO_MQT_REPO}
!python src/report_dataset_breakdown.py   --inputs     fixed_36={RESULT_FIXED_36}     fixed_64={RESULT_FIXED_64}     fixed_144={RESULT_FIXED_144}     fixed_256={RESULT_FIXED_256}     router_prompt={RESULT_ROUTER_PROMPT}     router_image={RESULT_ROUTER_IMAGE}     router_multimodal={RESULT_ROUTER_MULTIMODAL}     router_cross_attention={RESULT_ROUTER_CROSS_ATTN}   --out-dir {RESULTS_DIR}   --out-prefix {BREAKDOWN_PREFIX}   {EXCLUDE_DATASETS_ARGS}


/content/drive/Othercomputers/My Laptop/Auto-MQT-
wrote_csv: results/proposal_4ds_overall.csv
wrote_csv: results/proposal_4ds_by_dataset.csv
wrote_csv: results/proposal_4ds_paper_style.csv
wrote_markdown: results/proposal_4ds.md
wrote_figure: results/proposal_4ds_overall_score.png
wrote_figure: results/proposal_4ds_per_dataset_by_run.png
wrote_figure: results/proposal_4ds_fixed_budget_curves.png
wrote_figure: results/proposal_4ds_tradeoff.png
